In [2]:
from langchain.agents import create_agent
from langchain.agents.middleware import before_agent
from langchain.messages import SystemMessage

### 실습 문제 1. 한국어로만 답변하게 만드는 미들웨어 추가하기


In [9]:
@before_agent
def force_korean(state, runtime):
    return {
        'messages': [
            SystemMessage(content='모든 답변은 반드시 한국어로 작성하세요.')
        ] + state['messages']
    }

agent = create_agent(
    model='openai:gpt-5.4-mini',
    tools=[],
    middleware=[force_korean]
)

result = agent.invoke(
    {
        'messages': [
            {
                'role': 'user',
                'content': "Explain what RAG is. Answer briefly"
            }
        ]
    }
)

In [10]:
print(f"{result['messages'][-1].content}")

RAG는 **Retrieval-Augmented Generation**의 약자입니다.  
쉽게 말해, AI가 답을 만들기 전에 **관련 자료를 먼저 검색(Retrieval)** 하고, 그 내용을 바탕으로 **답변을 생성(Generation)** 하는 방식입니다.

장점:
- 더 최신 정보 반영 가능
- 환각(틀린 답) 줄이기
- 내부 문서나 DB를 활용하기 좋음

즉, **“검색 + 생성”을 결합한 AI 방식**입니다.


### 실습 문제 2. 금지어를 검사하는 미들웨어 추가하기

In [7]:
from typing import Any

from langchain.agents import create_agent, AgentState
from langchain.agents.middleware import before_model
from langchain.messages import AIMessage
from langgraph.runtime import Runtime

# 금지어 목록
banned_words = ["비밀번호", "주민번호", "욕설"]


@before_model(can_jump_to=["end"])
def block_banned_words(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    messages = state["messages"]

    # 가장 최근 사용자 메시지 찾기
    last_user_message = None
    for message in reversed(messages):
        if message.type == "human":
            last_user_message = message
            break

    if last_user_message is None:
        return None

    content = last_user_message.content

    # content가 문자열일 때만 검사
    if isinstance(content, str):
        for word in banned_words:
            if word in content:
                return {
                    "messages": [
                        AIMessage(
                            content="입력에 허용되지 않는 단어가 포함되어 있어 요청을 처리할 수 없습니다."
                        )
                    ],
                    "jump_to": "end",
                }

    return None


agent = create_agent(
    model="openai:gpt-5.4-nano",
    tools=[],
    middleware=[block_banned_words],
)

result = agent.invoke(
    {
        "messages": [
            {"role": "user", "content": "내 비밀번호는 1234야"}
        ]
    }
)

In [8]:
print(f"{result["messages"][-1].content}")

입력에 허용되지 않는 단어가 포함되어 있어 요청을 처리할 수 없습니다.


### 실습 문제 3. 답변 스타일을 조절하는 미들웨어 추가하기

In [11]:
from langchain.agents import create_agent, AgentState
from langchain.agents.middleware import before_model
from langchain.messages import SystemMessage
from langgraph.runtime import Runtime

# 여기에 미들웨어 함수를 작성하세요
@before_model
def adjust_response_style(state: AgentState, runtime: Runtime) -> dict[str, Any]:
    user_text = state['messages'][-1].content
    
    if len(user_text) <= 20:
        style_message = SystemMessage(content='짧고 간단하게 답하세요.')
    else:
        style_message = SystemMessage(content='예시를 포함해 자세히 설명하세요.')
    
    return {
        'messages': [style_message] + state['messages']
    }

agent = create_agent(
    model="openai:gpt-5.4-nano",
    tools=[],
    middleware=[adjust_response_style]
)

result = agent.invoke(
    {
        "messages": [
            {"role": "user", "content": "RAG란?"}
        ]
    }
)

In [12]:
print(result["messages"][-1].content)

RAG는 **Retrieval-Augmented Generation(검색 기반 생성)**의 약자예요.  

- **검색(Retrieval)**: 먼저 외부 문서/DB에서 관련 내용을 찾아와요.  
- **생성(Generation)**: 그 내용을 바탕으로 답변을 생성해요.  

즉, **모델이 지식을 “찾아온 뒤” 답을 만들기 때문에** 환각을 줄이고 최신/문서 기반 답변을 하는 데 유리해요.


### 실습 문제 4. 도구 호출 로그를 남기는 미들웨어 추가하기

In [1]:
from langchain.agents import create_agent, AgentState
from langchain.agents.middleware import wrap_tool_call
from langchain.tools import tool

from collections.abc import Callable
from langchain.messages import ToolMessage
from langchain.tools.tool_node import ToolCallRequest
from langgraph.types import Command


@tool
def add_numbers(a: int, b: int) -> str:
    """두 숫자를 더합니다."""
    return str(a + b)


@wrap_tool_call
def log_tool_calls(
    request: ToolCallRequest,
    handler: Callable[[ToolCallRequest], ToolMessage | Command],
) -> ToolMessage | Command:
    tool_name = request.tool_call['name']
    tool_args = request.tool_call['args']
    
    print(f"[TOOL START]{tool_name}/args={tool_args}")
    
    try:
        result = handler(request)

        if isinstance(result, ToolMessage):
            print(f"[TOOL END]{tool_name}/result={result.content}")
        else:
            print(f"[TOOL END]{tool_name}/result={result}")

        return result

    except Exception as e:
        print(f"[TOOL ERROR]{tool_name}/error={e}")
        raise


agent = create_agent(
    model="openai:gpt-5.4-nano",
    tools=[add_numbers],
    middleware=[log_tool_calls]
)

result = agent.invoke(
    {
        "messages": [
            {"role": "user", "content": "3과 5를 더해줘"}
        ]
    }
)

[TOOL START]add_numbers/args={'a': 3, 'b': 5}
[TOOL END]add_numbers/result=8


In [2]:
print(result["messages"][-1].content)

3과 5의 합은 **8**입니다.
